# Kaggle多语言基线 v4：明确固定T4 fp16
沿用原Dataset，兼容递归解压。T4训练明确采用fp16，不使用自动bf16能力判断。此版本用于尚未启动正式训练的会话。


In [ ]:
import time, sys, os, json, zipfile, shutil, subprocess, importlib.metadata
from pathlib import Path
SESSION_START = time.monotonic()
SESSION_HOURS = 10.5  # 整个本次运行的墙钟预算；剩余额度不足时主动调小
RESUME = ""  # 通常自动识别；多个旧包时填写希望恢复的唯一ZIP路径
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
assert WORK.is_dir(), "请在Kaggle运行这个Notebook"
BASE = WORK / "kaggle_multilang_baseline"
SOURCE = Path("/kaggle/input/datasets/<Kaggle用户名>/kaggle-multilang-baseline/kaggle_multilang_baseline")
assert SOURCE.is_dir(), "请确认运行包Dataset已添加"
shutil.copytree(SOURCE,BASE,dirs_exist_ok=True)
import hashlib
# Verify the original package files; official.zip was expanded by Kaggle.
manifest=json.loads((SOURCE/"bundle_manifest.json").read_text())
for n,h in manifest.items():
    if n=="official.zip": continue
    assert (SOURCE/n).is_file() and hashlib.sha256((SOURCE/n).read_bytes()).hexdigest()==h, f"原运行包文件变化：{n}"
RUNNER_V3 = '"""Independent T4 workers, immutable completed tasks, portable result archives."""\nimport argparse, hashlib, importlib.util, json, math, os, shutil, subprocess, sys, time, zipfile\nfrom pathlib import Path\nHERE=Path(__file__).resolve().parent\nREV=\'a0484667b22365f84929a935b5e50a51f71f159d\'\nUPSTREAM=\'83bfe561f0f5ab5f9ff52b37dd9619faf34d3ebcb7a34032a14fa55f9b597f63\'\n\ndef require(ok,msg):\n    if not ok: raise RuntimeError(msg)\ndef read(p): return json.loads(Path(p).read_text(encoding=\'utf-8\'))\ndef write(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    temp=p.with_suffix(p.suffix+\'.tmp\');temp.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding=\'utf-8\');temp.replace(p)\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\ndef identity(): return hashlib.sha256((sha(HERE/\'runner.py\')+sha(HERE/\'upstream_mdeberta.py\')+sha(HERE/\'official_files.json\')).encode()).hexdigest()\ndef tasks(): return [(t,s,f) for s in (42,43,44) for f in range(5) for t in (4,2,3)]\ndef tag(t): return \'t%s_s%s_f%s\'%tuple(t)\ndef target(root,t): return Path(root)/\'formal\'/tag(t)\ndef task_result(root,t,smoke=False):\n    tr,s,f=t\n    return Path(root)/(\'smoke\' if smoke else \'formal\')/tag(t)/f\'track{tr}/ce/seed{s}/fold{f}\'\ndef safe_extract(archive,dest):\n    dest=Path(dest).resolve()\n    with zipfile.ZipFile(archive) as z:\n        require(z.testzip() is None,\'ZIP损坏\')\n        for n in z.namelist(): require((dest/n).resolve().is_relative_to(dest),\'ZIP越界路径\')\n        z.extractall(dest)\n\ndef module(root,t=None,smoke=False,gpu=None):\n    root=Path(root).resolve()\n    visible=os.environ.get(\'CUDA_VISIBLE_DEVICES\',\'0,1\')\n    os.environ.update(DICO_WORK=str(root/\'work\'),DICO_DIR=str(root/\'official\'),DICO_STAGE=\'smoke\' if smoke else \'stage2\',DICO_PLATFORM=\'Kaggle\',HF_HOME=str(root/\'hf_cache\'))\n    os.environ.setdefault(\'HF_ENDPOINT\',\'https://huggingface.co\')\n    require(sha(HERE/\'upstream_mdeberta.py\')==UPSTREAM,\'原训练源码哈希不匹配\')\n    spec=importlib.util.spec_from_file_location(\'baseline_upstream\',HERE/\'upstream_mdeberta.py\');m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m)\n    # Upstream sets CUDA_VISIBLE_DEVICES=0 at import. Restore worker assignment BEFORE importing torch.\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=str(gpu) if gpu is not None else visible\n    m.OUT=str(root/(\'smoke\' if smoke else \'formal\')/tag(t)) if t else str(root/\'aggregate\')\n    original=m.common_fingerprint\n    m.common_fingerprint=lambda:{**original(),\'multilang_driver\':identity()}\n    return m\n\ndef environment():\n    import torch, importlib.metadata as md\n    require(torch.cuda.is_available(),\'没有GPU\')\n    require(\'T4\' in torch.cuda.get_device_name(0),\'本包冻结为T4，不混合其他GPU\')\n    return {\'gpu\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\n            \'python\':sys.version.split()[0],**{k:md.version(k) for k in (\'transformers\',\'numpy\',\'sentencepiece\',\'huggingface_hub\')}}\n\ndef validate_task(root,t,m=None):\n    d=task_result(root,t);m=m or module(root,t)\n    done=read(d/\'done.json\');require(read(d/\'task.json\')==m.task_fingerprint(m.make_task(t[0],\'ce\',t[1],t[2],t[2]==0),REV),\'任务指纹变化\')\n    require(done[\'code_sha256\']==identity() and done[\'precision\']==\'fp16 autocast\',\'任务代码/精度不符\')\n    require(done[\'env\'][\'runtime\']==read(Path(root)/\'runtime.json\'),\'任务运行环境不同\')\n    require(len(read(d/\'history.json\'))==12,\'未完成12轮\')\n    manifest=read(d/\'verified.json\')\n    require(all((d/n).is_file() and sha(d/n)==h for n,h in manifest.items()),\'任务文件缺失或变化\')\n    return done\n\ndef worker(root,t,gpu,smoke):\n    m=module(root,t,smoke,gpu);env=environment()\n    require(env==read(Path(root)/\'runtime.json\'),\'环境与当前正式组不一致\')\n    import torch\n    require(torch.cuda.device_count()==1,\'每个训练进程必须只看到一张T4\')\n    print(\'物理GPU\',gpu,\'；可见卡数\',torch.cuda.device_count(),\'；卡型\',torch.cuda.get_device_name(0),flush=True)\n    # Freeze intended T4 precision instead of relying on version-dependent capability detection.\n    m.amp_dtype=lambda: torch.float16\n    print("固定训练精度：",m.amp_dtype(),flush=True)\n    torch.set_num_threads(2)\n    m._MODEL_PATH[\'path\']=read(Path(root)/\'model.json\')[\'path\']\n    m.ENV.update({**m.env_info(),\'platform\':\'Kaggle\',\'amp\':\'fp16\',\'runtime\':env,\'physical_gpu\':gpu})\n    m.check_common_fingerprint()\n    task=m.make_task(t[0],\'ce\',t[1],t[2],t[2]==0)\n    original=m.TRAIN_FN\n    def strict(*a,**kw):\n        result=original(*a,**kw)\n        require(result.get(\'precision\')==\'fp16 autocast\',f"训练精度不符：实际={result.get(\'precision\')!r}，尝试记录={result.get(\'precision_attempts\')!r}；要求fp16 autocast")\n        return result\n    m.TRAIN_FN=strict\n    m.run_task(task,REV,identity())\n    d=task_result(root,t,smoke)\n    done=read(d/\'done.json\')\n    require(done[\'precision\']==\'fp16 autocast\',\'错误精度\')\n    if smoke:\n        full=m.parts_for(t[0],t[2],True,smoke=False)\n        timing=original(task,full,max_steps=12)\n        require(\'failed\' not in timing,\'正式规模测速失败\')\n        step=sum(timing[\'step_times\'][3:])/len(timing[\'step_times\'][3:])\n        inference=timing[\'infer_seconds_per_input\']\n        estimate=(math.ceil(len(full[\'fit\'])/m.BATCH_SIZE)*12*step+12*(len(m.needed_inputs(full[\'val\'],both=False))*inference+timing[\'clone_seconds\'])+sum(len(m.needed_inputs(full[k])) for k in (\'test\',\'dev\'))*inference+timing[\'load_seconds\'])/60\n        write(Path(root)/\'timing\'/f\'track{t[0]}.json\',{\'minutes_estimate\':estimate,\'seconds_per_step\':step,\'timing\':timing})\n    write(d/\'verified.json\',{p.relative_to(d).as_posix():sha(p) for p in sorted(d.rglob(\'*\')) if p.is_file() and p.name!=\'verified.json\'})\n\ndef prepare(root,resume):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    expected=read(HERE/\'official_files.json\')\n    candidates=[p for p in (HERE/\'official\').iterdir() if p.is_dir() and (p/\'evaluation_functions/__init__.py\').is_file() and (p/\'final_data\').is_dir()]\n    require(len(candidates)==1,\'已解压官方数据目录不唯一\')\n    source=candidates[0]\n    for n,h in expected.items():\n        require((source/n).is_file() and sha(source/n)==h,\'官方数据或评分文件不同: \'+n)\n    if not (root/\'official\').exists():shutil.copytree(source,root/\'official\')\n    for n,h in expected.items():\n        require((root/\'official\'/n).is_file() and sha(root/\'official\'/n)==h,\'工作目录数据变化: \'+n)\n    if resume:\n        with zipfile.ZipFile(resume) as z:\n            meta=json.loads(z.read(\'MANIFEST.json\'));require(meta[\'identity\']==identity(),\'恢复包代码/数据版本不同\')\n            require(z.testzip() is None,\'恢复包损坏\')\n            for n,h in meta[\'files\'].items():\n                require((root/n).resolve().is_relative_to(root.resolve()),\'恢复路径越界\')\n                b=z.read(n);require(hashlib.sha256(b).hexdigest()==h,\'恢复文件哈希错误\')\n                # Only import completed tasks and frozen environment. Session clocks/SMOKE are not portable.\n                if n.startswith(\'formal/\') or n==\'runtime.json\':\n                    p=root/n\n                    if p.exists():require(p.read_bytes()==b,\'已有文件与恢复包冲突\')\n                    else:p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(b)\n    m=module(root)\n    env=environment()\n    import torch\n    require(torch.cuda.device_count()==2 and all(\'T4\' in torch.cuda.get_device_name(i) for i in range(2)),\'请选择T4×2\')\n    if (root/\'runtime.json\').exists():require(read(root/\'runtime.json\')==env,\'恢复环境与原组不同，先核对依赖，不混入结果\')\n    write(root/\'runtime.json\',env)\n    path,revision,source=m.fetch_model(REV);require(revision==REV,\'模型版本变化\')\n    write(root/\'model.json\',{\'path\':path,\'revision\':revision,\'source\':source})\n    for t in tasks():\n        if (task_result(root,t)/\'verified.json\').exists():validate_task(root,t,module(root,t))\n    print(\'准备完成；45项CE基线，T4×2独立任务。\',flush=True)\n\ndef pack(root):\n    root=Path(root);temp=root/\'results.zip.tmp\';files={}\n    with zipfile.ZipFile(temp,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for base in (\'formal\',\'smoke\',\'timing\',\'logs\',\'aggregate\'):\n            for p in sorted((root/base).rglob(\'*\')):\n                if not p.is_file() or p.suffix==\'.tmp\':continue\n                if base==\'formal\':\n                    job=p.relative_to(root/\'formal\').parts[0]\n                    if not list((root/\'formal\'/job).glob(\'track*/ce/seed*/fold*/verified.json\')):continue\n                n=p.relative_to(root).as_posix();b=p.read_bytes();z.writestr(n,b);files[n]=hashlib.sha256(b).hexdigest()\n        for n in (\'runtime.json\',\'status.json\',\'COMPLETE.json\',\'smoke_ok.json\'):\n            if (root/n).exists():\n                b=(root/n).read_bytes();z.writestr(n,b);files[n]=hashlib.sha256(b).hexdigest()\n        z.writestr(\'MANIFEST.json\',json.dumps({\'identity\':identity(),\'files\':files}))\n    dest=root/\'kaggle_multilang_results.zip\';temp.replace(dest)\n    print(\'已保存结果包：\',dest,flush=True)\n\ndef collect(root):\n    root=Path(root);m=module(root);out={};complete_tasks=[]\n    for t in tasks():\n        if not (task_result(root,t)/\'verified.json\').exists():continue\n        validate_task(root,t,module(root,t));complete_tasks.append(t)\n    m.OUT=str(root/\'aggregate\');fam,folds=m.split_base()\n    for tr in (2,3,4):\n        rows=m.read_rows(m.data_file(tr,\'train\',\'reference\'))\n        for seed in (42,43,44):\n            if not all((tr,seed,f) in complete_tasks for f in range(5)):continue\n            maps=[m.load_logits(str(task_result(root,(tr,seed,f))/\'logits_test.json\')) for f in range(5)]\n            require(all(m.all_finite(x) for x in maps),\'OOF存在非有限值\')\n            d=root/\'aggregate\'/f\'track{tr}/seed{seed}\';scores={}\n            for mode in (\'plain\',\'sym\'):\n                labels=[m.argmax_label(m.probs(maps[folds[fam[r[\'pair_id\']]]],r[\'text1\'],r[\'text2\'],mode)) for r in rows]\n                scores[mode]=m.official_score(rows,labels,str(d),\'oof_\'+mode)\n            out[f\'track{tr}/seed{seed}\']={\'rows\':len(rows),\'scores\':scores}\n    write(root/\'aggregate/summary.json\',out)\n    if len(complete_tasks)==45:\n        require(len(out)==9,\'OOF数量错误\');write(root/\'COMPLETE.json\',{\'tasks\':45,\'oof\':out,\'identity\':identity()})\n        print(\'COMPLETE：45项正式任务、9套完整OOF。\',flush=True)\n    return len(complete_tasks)\n\ndef enough_time(now,deadline,estimate): return now+max(600,estimate*60*1.6)<deadline\n\ndef queue(root,smoke,hours):\n    root=Path(root)\n    if not smoke:require(read(root/\'smoke_ok.json\')[\'identity\']==identity(),\'请先完成本次会话SMOKE\')\n    deadline=time.monotonic()+hours*3600\n    pending=[(2,42,0),(3,42,0),(4,42,0)] if smoke else tasks()\n    pending=[t for t in pending if smoke or not (task_result(root,t)/\'verified.json\').exists()]\n    active={};failed=[];closed=False\n    try:\n        while pending or active:\n            for gpu in (0,1):\n                if gpu in active:continue\n                if not pending or closed:continue\n                # Complete earlier seed first; both GPUs may work on that seed only.\n                seed=pending[0][1]\n                if any(t[0][1]<seed for t in active.values()):continue\n                t=pending[0]\n                estimate=read(root/\'timing\'/f\'track{t[0]}.json\')[\'minutes_estimate\'] if not smoke else 10\n                if not enough_time(time.monotonic(),deadline,estimate):closed=True;pending=[];break\n                pending.pop(0);log=root/\'logs\'/f\'{"smoke" if smoke else "run"}_{tag(t)}.log\';log.parent.mkdir(exist_ok=True)\n                f=log.open(\'a\',encoding=\'utf-8\');env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n                p=subprocess.Popen([sys.executable,\'-u\',str(HERE/\'runner.py\'),\'worker\',\'--root\',str(root),\'--task\',*map(str,t),\'--gpu\',str(gpu)]+([\'--smoke\'] if smoke else []),env=env,stdout=f,stderr=subprocess.STDOUT)\n                active[gpu]=(t,p,f);print(\'开始\',tag(t),\'GPU\',gpu,\'日志\',log,flush=True)\n            for gpu,(t,p,f) in list(active.items()):\n                if p.poll() is None:continue\n                f.close();del active[gpu]\n                if p.returncode:failed.append(tag(t));print(\'失败\',tag(t),\'请查看日志\',flush=True)\n                else:print(\'完成\',tag(t),flush=True)\n                pack(root)\n            if active:time.sleep(3)\n        if smoke:\n            require(not failed and not closed,\'SMOKE未全部完成；不允许正式训练\')\n            write(root/\'smoke_ok.json\',{\'identity\':identity(),\'runtime\':read(root/\'runtime.json\')})\n            print(\'SMOKE通过，三个赛道测速：\', {t:read(root/\'timing\'/f\'track{t}.json\')[\'minutes_estimate\'] for t in (2,3,4)},flush=True)\n        else:\n            n=collect(root);write(root/\'status.json\',{\'completed\':n,\'total\':45,\'failed\':failed,\'time_limit_reached\':closed})\n            print(\'本会话完成；累计\',n,\'/45；失败\',failed,flush=True)\n    finally:\n        for t,p,f in active.values():\n            p.terminate();p.wait();f.close()\n        pack(root)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'command\',choices=[\'prepare\',\'smoke\',\'run\',\'worker\',\'check\',\'pack\'])\n    p.add_argument(\'--root\',default=\'/kaggle/working/dico_multilang\');p.add_argument(\'--resume\')\n    p.add_argument(\'--hours\',type=float,default=10.5);p.add_argument(\'--task\',type=int,nargs=3);p.add_argument(\'--gpu\',type=int,choices=[0,1]);p.add_argument(\'--smoke\',action=\'store_true\');a=p.parse_args()\n    root=Path(a.root).resolve();root.mkdir(parents=True,exist_ok=True)\n    if a.command==\'worker\':\n        require(tuple(a.task) in tasks(),\'非法任务\');worker(root,tuple(a.task),a.gpu,a.smoke);return\n    import fcntl\n    with (root/\'queue.lock\').open(\'w\') as lock:\n        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n        if a.command==\'prepare\':prepare(root,a.resume)\n        elif a.command in (\'smoke\',\'run\'):queue(root,a.command==\'smoke\',a.hours)\n        elif a.command==\'check\':collect(root);pack(root)\n        else:pack(root)\n\nif __name__==\'__main__\':main()\n'
OFFICIAL_FILES = {'LICENSE': '3972dc9744f6499f0f9b2dbf76696f2ae7ad8af9b23dde66d6af86c9dfb36986', 'README.md': 'e263bca7772e005b811e44ada4f77711e8157ee541bbce7ec80c2f4f9b81220b', 'evaluation_functions/README.md': '39901ba138b27d5d0ae3af739e3314915a6db51d1b9ce04dc06dd9e49ad7b23d', 'evaluation_functions/__init__.py': '5e5ff6d6fd58b01638a8fd7ee7f482431bff837d134a0e79e5bfe0a5705baa15', 'evaluation_functions/__main__.py': 'f27cadcf05f172d2b853cb371ab3f99b098287f0ddbe0e19179df8a1143962fa', 'evaluation_functions/aggregation.py': 'a8ab594ca319c5c1eeeaa7307e427d028614cb65e37d5e95f2efaf60dfac4fc0', 'evaluation_functions/cli.py': '00062311915e28170b71b4a524cf270381c6b8256660c45f3d182d603cb95540', 'evaluation_functions/errors.py': '00173f95a00c80e8190a8e73f2b452856c11a8eaaee72608a03cffb47f2cb128', 'evaluation_functions/io.py': 'a87e368e154c4cd7350142d1af5a4c83041a81f68d18dbbc9b5f5bc2297c4cd1', 'evaluation_functions/labels.py': 'd500de7a8ef68460322516e4cbf431e19387e7555e9a4c2b2673dd727b17a98f', 'evaluation_functions/metrics.py': '448c0fd72b149b091216b9a65bfbf88b3744aa124f20e5c241c51ff33f7efd7e', 'evaluation_functions/schemas.py': '3a8cbca1ae93ba547d23bcf9c8eec0de2cf803f5fdc63166e7ad3fbb243c3193', 'evaluation_functions/scorer.py': '076c5956e82fffe6234f1cecb3351f555c92f3d61798eab30f68e0116a7c038f', 'evaluation_functions/templates.py': 'ded61fc53ddbdebd25f27ed01730012693f742dd210654d61392603ce4892c27', 'evaluation_functions/validation.py': 'bb6258b41b1312fc88e5851bf9bb8615694c3600b0ef73643bc1cdff7720b9ea', 'final_data/README.md': '42cd775ea44b6f2dc9909a9e2cf5ede1cad168d7a60186fd21491d10599e7d35', 'final_data/dev/dico_nli_dev_track1_participant_labeled.csv': '80b7152f7ccf0d349d42ad2f098988106689b918b36337e5756b121d17513511', 'final_data/dev/dico_nli_dev_track1_reference.csv': '8d5cd11c1c60df3c9b59254d12a60fb2cf43e91dde01b46c8a8f296b806134bc', 'final_data/dev/dico_nli_dev_track1_submission_template.csv': '28541a99b346cb7ab514b76bbf37d95e86be26bb3679c040fd1ab538824baa52', 'final_data/dev/dico_nli_dev_track2_participant_labeled.csv': '09a08a2ffdbb438ead917457f05d977233436431939d7859abd4bf6fe5ee446b', 'final_data/dev/dico_nli_dev_track2_reference.csv': 'd962e90cbf221cbc5cd6fc197edcb423be3b5ba5f08acd80b8b0a70d27a50171', 'final_data/dev/dico_nli_dev_track2_submission_template.csv': '604685ed59e3aaf9e4709596dddb5c2407b332c50103f7ca3db4180896c816f7', 'final_data/dev/dico_nli_dev_track3_participant_labeled.csv': 'c42f051c8f5908cceab4c8f80b003c2314d0c13a487be8586011a7ae3b67493c', 'final_data/dev/dico_nli_dev_track3_reference.csv': '3ea3643a5c6e0b2ca4606e3935cd7b368b0fd8f157691c80a5c8d5a3745cfebe', 'final_data/dev/dico_nli_dev_track3_submission_template.csv': '7ca0856a4a6a55b3b1d6d255b75f05cabb521f44fe2c375573ea4fb590f83f6f', 'final_data/dev/dico_nli_dev_track4_participant_labeled.csv': '62fd2b4180c03ab0389bd9348d5c81b04b13b936b325a4a785d0c4c8d6c03baf', 'final_data/dev/dico_nli_dev_track4_reference.csv': '1d2769d1e757eb2171272c8e56e4cb7593f7f05de826c4b22c2efb6d3e2fc9ae', 'final_data/dev/dico_nli_dev_track4_submission_template.csv': '6bfd6c7bf1ad7ac259c22ea4c10311537856f44c0f4b49ee616a9072ecf95830', 'final_data/train/dico_nli_train_track1_participant_labeled.csv': '5e8dd9f3a7e000933fec3c9e64d064f9a25586f141f58d6e6afb47c4fc797211', 'final_data/train/dico_nli_train_track1_reference.csv': 'c8f4d8587711e0beaefc115606e5c29c3e78dc6660e074967428af0fed069ee4', 'final_data/train/dico_nli_train_track1_submission_template.csv': 'd38ee22049b79fd2030a96fca68c7df4f04ef37aac91a6868b2dc6e9661905e8', 'final_data/train/dico_nli_train_track2_participant_labeled.csv': '68bfbcdd12bffc93fdc84a6526fbb217846d657faf9dbce766a9297cccade163', 'final_data/train/dico_nli_train_track2_reference.csv': '9850c08f3027961df47a2635a961902e4d5f99a6c57e8a04049e69b52b571e01', 'final_data/train/dico_nli_train_track2_submission_template.csv': '0cc284a7cb6bb15899e396f479e2aedfcac17ba3bf93929fd5d7006c448a582c', 'final_data/train/dico_nli_train_track3_participant_labeled.csv': '5d085cdd555800285c2c6f35865713d142034b804dbf2e8c349a37bad90d4cfa', 'final_data/train/dico_nli_train_track3_reference.csv': '32f03eb1ec0fa079b8fcec8e269e9aade5cd04c735d1873da88fe0f403dfeef3', 'final_data/train/dico_nli_train_track3_submission_template.csv': '7024a186a3fcb509efa89f10b05078d484214df3e7c24d70bad74f7cd462425b', 'final_data/train/dico_nli_train_track4_participant_labeled.csv': 'f60ad9359c69b52fb3683170c198e781c83e26c06e6a8078c1e3d033b8a9c948', 'final_data/train/dico_nli_train_track4_reference.csv': '3626a1d3d47b983ea8a0fe4cdeecfca6c9b537c31e4b5131c354c1a592544ce9', 'final_data/train/dico_nli_train_track4_submission_template.csv': '365aa7d3f503ff62453f4ff12b2317242780331d32ada7a911761b626cf34486'}
(BASE/"runner.py").write_text(RUNNER_V3,encoding="utf-8")
(BASE/"official_files.json").write_text(json.dumps(OFFICIAL_FILES,sort_keys=True,indent=2),encoding="utf-8")
roots=[p for p in (BASE/"official").iterdir() if p.is_dir() and (p/"evaluation_functions/__init__.py").is_file() and (p/"final_data").is_dir()]
assert len(roots)==1, f"官方仓库根目录不唯一：{roots}"
for n,h in OFFICIAL_FILES.items():
    assert (roots[0]/n).is_file() and hashlib.sha256((roots[0]/n).read_bytes()).hexdigest()==h, f"官方文件变化：{n}"
print("已验证官方文件",len(OFFICIAL_FILES),"个；嵌套ZIP及其展开副本不作为训练输入。")
version=importlib.metadata.version("torch").split("+")[0]
assert tuple(map(int,version.split(".")[:2])) >= (2,6), "镜像torch低于2.6，请把版本发给GPT确认安装，不直接混用旧环境"
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==5.19.0","huggingface_hub==2.2.0","sentencepiece==0.2.2","protobuf>=4.25,<8"],check=True)
os.environ.update(HF_ENDPOINT="https://huggingface.co",HF_HUB_DISABLE_XET="1",TOKENIZERS_PARALLELISM="false",CUDA_VISIBLE_DEVICES="0,1")
ROOT=WORK/"dico_multilang"
def run(command, *args):
    subprocess.run([sys.executable,"-u",str(BASE/"runner.py"),command,"--root",str(ROOT),*map(str,args)],check=True)
print("运行包准备完毕。")


In [ ]:
# 恢复只接收本包的T4结果，不接收历史A10包。
if not RESUME:
    previous=list(INPUT.rglob("kaggle_multilang_results*.zip"))
    unpacked=[p for p in INPUT.rglob("MANIFEST.json") if "identity" in json.loads(p.read_text()) and "files" in json.loads(p.read_text())]
    assert len(previous)+len(unpacked)<=1, "检测到多个恢复包，请在RESUME中明确指定唯一ZIP"
    if previous: RESUME=str(previous[0])
    elif unpacked:
        folder=unpacked[0].parent; manifest=json.loads(unpacked[0].read_text())
        rebuilt=WORK/"restored_input.zip"
        with zipfile.ZipFile(rebuilt,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(unpacked[0],"MANIFEST.json")
            for n in manifest["files"]:
                assert (folder/n).resolve().is_relative_to(folder.resolve())
                z.write(folder/n,n)
        RESUME=str(rebuilt)
run("prepare", *(["--resume",RESUME] if RESUME else []))


## SMOKE：两张卡、三个赛道

运行小规模训练、官方评分和正式batch测速。通过后查看每个赛道估计时长；它只是估计，不保证能在本次会话跑完45项。若出错，保留结果和日志发给GPT，不继续运行正式单元格。


In [ ]:
run("smoke", "--hours", max(0.01,SESSION_HOURS-(time.monotonic()-SESSION_START)/3600))


## 正式运行

按seed42→43→44逐步完成三赛道五折。动态领取任务；临近会话期限停止领取新任务，已有任务继续完成。每项完成后自动打包。不可保证强制关机前正在运行的任务一定结束；未完成的单项下次从头训练。

Notebook执行期间保持进程运行；需要后台运行时使用Kaggle的Save Version / Save & Run All流程，并确保已检查SMOKE。不要同时启动第二份Notebook训练同一输出目录。


In [ ]:
remaining=SESSION_HOURS-(time.monotonic()-SESSION_START)/3600
assert remaining>0, "本次会话预算已用完，请下载输出后换新会话"
run("run","--hours",remaining)


## 保存结果并继续下一会话

输出 `dico_multilang/kaggle_multilang_results.zip`。看到 COMPLETE 才代表45项和9套OOF全部完成；否则下载部分结果包，下一会话作为Input恢复。务必保存Kaggle输出或下载ZIP，/kaggle/working不会自动随新会话保留。

完成种子/折的预测与配置可恢复；本包沿用原编码器脚本，不输出模型权重或训练中途检查点。训练被强制打断时，只重做该单项，不重做已完成项。


In [ ]:
from IPython.display import FileLink, display
print((ROOT/"status.json").read_text() if (ROOT/"status.json").exists() else "请查看日志状态")
display(FileLink(str(ROOT/"kaggle_multilang_results.zip")))
